In [2]:
import pandas as pd
import numpy as np

import keras
import pickle

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_squared_error

In [3]:
df_train = pd.read_csv(r'../data/cleaned/train.csv')
df_test = pd.read_csv(r'../data/cleaned/test.csv')
df_val = pd.read_csv(r'../data/cleaned/val.csv')

In [4]:
def create_sequences(df,seq_length, target_col='PJME_MW',horizon=1):
    """
    Converts a scaled DataFrame into 3D sequence arrays for Keras models.
    """
    # Separate feature columns from Datetime
    feature_cols = [col for col in df.columns if col != 'Datetime']
    target_idx = feature_cols.index(target_col)
    
    data = df[feature_cols].values
    X, y = [], []
    
    for i in range(len(data) - seq_length - horizon + 1):
        X.append(data[i : i + seq_length, :])
        if horizon == 1:
            y.append(data[i + seq_length, target_idx])
        else:
            y.append(data[i + seq_length : i + seq_length + horizon, target_idx])
            
    return np.array(X), np.array(y)

In [5]:
SEQ_LEN = 24
HORIZON = 24

X_train, y_train = create_sequences(df_train, seq_length=SEQ_LEN, horizon=HORIZON)
X_val, y_val     = create_sequences(df_val, seq_length=SEQ_LEN, horizon=HORIZON)
X_test, y_test   = create_sequences(df_test, seq_length=SEQ_LEN, horizon=HORIZON)

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape:  {X_test.shape}")

X_train Shape: (101593, 24, 13)
X_test Shape:  (21733, 24, 13)


In [6]:
with open('target_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [7]:
y_test_mw = scaler.inverse_transform(y_test.reshape(-1, 1))

In [8]:
def evalution(model,X_test,y_test_mw): 
    y_pred_scaled = model.predict(X_test)
    y_pred_mw = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1))
    mae = mean_absolute_error(y_test_mw, y_pred_mw)
    rmse = np.sqrt(mean_squared_error(y_test_mw, y_pred_mw))
    mape = np.mean(np.abs((y_test_mw - y_pred_mw) / y_test_mw)) * 100
    r2 = r2_score(y_test_mw, y_pred_mw)
    mse = mean_squared_error(y_test_mw, y_pred_mw)
    
    return mae,mse,rmse,mape,r2



In [9]:
performance ={"Model":[],
              "MAE":[],
              "MSE":[],
              "RMSE":[],
              "MAPE":[],
              "R2":[]}

In [10]:
def add(name,mae,mse,rmse,mape,r2):
    performance['Model'].append(name)
    performance['MAE'].append(round(mae,2))
    performance['MSE'].append(round(mse,2))
    performance['RMSE'].append(round(rmse,2))
    performance['MAPE'].append(round(mape,2))
    performance['R2'].append(round(r2*100,2))
    
    return performance

In [11]:
add('Navie',1054.66,1834852.83,1354.57,3.43,0.96)

{'Model': ['Navie'],
 'MAE': [1054.66],
 'MSE': [1834852.83],
 'RMSE': [1354.57],
 'MAPE': [3.43],
 'R2': [96.0]}

In [12]:
add("Moving Average",3946.08,25384046.61,5038.26,13.13,0.39)

{'Model': ['Navie', 'Moving Average'],
 'MAE': [1054.66, 3946.08],
 'MSE': [1834852.83, 25384046.61],
 'RMSE': [1354.57, 5038.26],
 'MAPE': [3.43, 13.13],
 'R2': [96.0, 39.0]}

In [13]:
model =  keras.models.load_model(r"../models/lr_rnn.keras")

In [14]:
mae,mse,rmse,mape,r2 = evalution(model,X_test,y_test_mw)

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step


In [15]:
add('RNN',mae,mse,rmse,mape,r2)

{'Model': ['Navie', 'Moving Average', 'RNN'],
 'MAE': [1054.66, 3946.08, 1464.36],
 'MSE': [1834852.83, 25384046.61, 4151167.52],
 'RMSE': [1354.57, 5038.26, np.float64(2037.44)],
 'MAPE': [3.43, 13.13, np.float64(4.68)],
 'R2': [96.0, 39.0, 90.0]}

In [16]:
#lstm

In [17]:
model = keras.models.load_model(r'../models/lstm_early.keras')

In [18]:
mae,mse,rmse,mape,r2 = evalution(model,X_test,y_test_mw)

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


In [19]:
add('LSTM',mae,mse,rmse,mape,r2)

{'Model': ['Navie', 'Moving Average', 'RNN', 'LSTM'],
 'MAE': [1054.66, 3946.08, 1464.36, 1395.0],
 'MSE': [1834852.83, 25384046.61, 4151167.52, 3831021.71],
 'RMSE': [1354.57, 5038.26, np.float64(2037.44), np.float64(1957.3)],
 'MAPE': [3.43, 13.13, np.float64(4.68), np.float64(4.4)],
 'R2': [96.0, 39.0, 90.0, 90.77]}

In [20]:
#gru

In [21]:
model = keras.models.load_model(r'../models/gru_16.keras')

In [22]:
mae,mse,rmse,mape,r2 = evalution(model,X_test,y_test_mw)

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


In [23]:
add('GRU',mae,mse,rmse,mape,r2)

{'Model': ['Navie', 'Moving Average', 'RNN', 'LSTM', 'GRU'],
 'MAE': [1054.66, 3946.08, 1464.36, 1395.0, 1338.7],
 'MSE': [1834852.83, 25384046.61, 4151167.52, 3831021.71, 3669936.61],
 'RMSE': [1354.57,
  5038.26,
  np.float64(2037.44),
  np.float64(1957.3),
  np.float64(1915.71)],
 'MAPE': [3.43, 13.13, np.float64(4.68), np.float64(4.4), np.float64(4.18)],
 'R2': [96.0, 39.0, 90.0, 90.77, 91.16]}

In [24]:
#bilstm

In [25]:
model = keras.models.load_model(r'../models/bilstm_lr.keras')

In [26]:
SEQ_LEN = 168
HORIZON = 24

X_train168, y_train168 = create_sequences(df_train, seq_length=SEQ_LEN, horizon=HORIZON)
X_val168, y_val168     = create_sequences(df_val, seq_length=SEQ_LEN, horizon=HORIZON)
X_test168, y_test168   = create_sequences(df_test, seq_length=SEQ_LEN, horizon=HORIZON)

print("X_train:", X_train168.shape)
print("y_train:", y_train168.shape)

print("X_test:", X_test168.shape)
print("y_test:", y_test168.shape)

X_train: (101449, 168, 13)
y_train: (101449, 24)
X_test: (21589, 168, 13)
y_test: (21589, 24)


In [27]:
y_test_mw = scaler.inverse_transform(y_test168.reshape(-1, 1))

In [28]:
mae,mse,rmse,mape,r2 = evalution(model,X_test168,y_test_mw)

675/675 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step


In [29]:
add('BILSTM',mae,mse,rmse,mape,r2)

{'Model': ['Navie', 'Moving Average', 'RNN', 'LSTM', 'GRU', 'BILSTM'],
 'MAE': [1054.66, 3946.08, 1464.36, 1395.0, 1338.7, 1457.88],
 'MSE': [1834852.83,
  25384046.61,
  4151167.52,
  3831021.71,
  3669936.61,
  4004100.32],
 'RMSE': [1354.57,
  5038.26,
  np.float64(2037.44),
  np.float64(1957.3),
  np.float64(1915.71),
  np.float64(2001.02)],
 'MAPE': [3.43,
  13.13,
  np.float64(4.68),
  np.float64(4.4),
  np.float64(4.18),
  np.float64(4.64)],
 'R2': [96.0, 39.0, 90.0, 90.77, 91.16, 90.35]}

In [30]:
df = pd.DataFrame(performance)

In [31]:
df

,Model,MAE,MSE,RMSE,MAPE,R2
0,Navie,1054.66,1834852.83,1354.57,3.43,96.00
1,Moving Average,3946.08,25384046.61,5038.26,13.13,39.00
2,RNN,1464.36,4151167.52,2037.44,4.68,90.00
3,LSTM,1395.00,3831021.71,1957.30,4.40,90.77
4,GRU,1338.70,3669936.61,1915.71,4.18,91.16
5,BILSTM,1457.88,4004100.32,2001.02,4.64,90.35


In [32]:
df.to_csv(r'../data/cleaned/performance.csv',index=False)